In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "dualtest_experiments" else Path.cwd()
DUALTEST_DIR = PROJECT_ROOT / "DUALTEST"

sys.path.append(str(DUALTEST_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DUALTEST_DIR:", DUALTEST_DIR)

PROJECT_ROOT: /home/alumno1/Desktop/NLP_Proyecto_Final-main
DUALTEST_DIR: /home/alumno1/Desktop/NLP_Proyecto_Final-main/DUALTEST


In [ ]:
import pandas as pd
import torch
import inspect

from tqdm.auto import tqdm
from target_model import HFLocalTarget
from experiment_utils import prepare_records
from transformers import AutoTokenizer
from prefixing import split_by_tokens
from target_model import HFLocalTarget
from reference_model import ReferenceModel
from metrics import run_length, edit_similarity, rlb_score, esb_score



In [ ]:
records = prepare_records(
    dataset_name="booktection",
    n=3,
    random_state=7,
    balance_labels=True,
)

len(records), records[0].keys()

(3,
 dict_keys(['id', 'dataset', 'dataset_family', 'source_dataset', 'file_path', 'label', 'estimated_membership', 'text', 'text_hash']))

In [4]:
record = records[0]

print("ID:", record["id"])
print("Label:", record["label"])
print("Membership:", record["estimated_membership"])
print("Texto:")
print(record["text"][:1000])

ID: booktection_13717_A.txt
Label: 0
Membership: non_member
Texto:
I stared at Clara, wanting to ask her questions, perhaps wanting to embrace her, but it was too late now, for she was handing me a pair of white gloves. Her eyes gleamed with a jealousy I could scarcely credit from my twin, who had always had the world at her feet. Drosselmeyer had created her to protect himself, and so she should have danced through the world all her life, clear and bright and untouched, just as I should have been relegated to corners and alcoves, the dark sister, knowing a future I was unable to change. But magic was only magic, as Anastasia had always said. Thinking of Anastasia reminded me of Fritz, of that odd day on the stairs, and I grasped Clara’s elbow.


In [ ]:
reference_model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(reference_model_name)

split = split_by_tokens(
    record["text"],
    tokenizer=tokenizer,
    prefix_len=64,
    continuation_len=64,
)

print("PREFIJO:")
print(split.prefix_text)

print("\nCONTINUACIÓN REAL:")
print(split.continuation_text)

/home/alumno1/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PREFIJO:
I stared at Clara, wanting to ask her questions, perhaps wanting to embrace her, but it was too late now, for she was handing me a pair of white gloves. Her eyes gleamed with a jealousy I could scarcely credit from my twin, who had always had the world at her feet. Drosselmeyer

CONTINUACIÓN REAL:
 had created her to protect himself, and so she should have danced through the world all her life, clear and bright and untouched, just as I should have been relegated to corners and alcoves, the dark sister, knowing a future I was unable to change. But magic was only magic, as Anastasia had always said


## Prueba de funcionamiento

In [ ]:
print("CUDA:", torch.cuda.is_available())
print(torch.version.cuda)

CUDA: True
13.0


In [ ]:
print(inspect.signature(HFLocalTarget.__init__))

(self, model_name: str, device: None, dtype=torch.bfloat16)


In [9]:
target_model_name = "Qwen/Qwen2.5-1.5B"
reference_model_name = "Qwen/Qwen2.5-0.5B"

target = HFLocalTarget(target_model_name, device= "cuda")
reference = ReferenceModel(reference_model_name, device= "cuda")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 11235.95it/s]


In [10]:
completion = target.complete(
    split.prefix_text,
    max_new_tokens=64,
    do_sample=False
)

print("COMPLETION TARGET:")
print(completion.text)

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


COMPLETION TARGET:
 had given her a pair of gloves, and I had been so pleased to see her wearing them that I had not noticed that she had not given me a pair. I had been so pleased to see her wearing them that I had not noticed that she had not given me a pair. I had been so pleased to see


In [11]:
target_tokens = completion.token_ids
source_tokens = split.continuation_token_ids
prefix_tokens = split.prefix_token_ids

print("Tokens target:", len(target_tokens))
print("Tokens source:", len(source_tokens))
print("Tokens prefix:", len(prefix_tokens))

Tokens target: 64
Tokens source: 64
Tokens prefix: 64


In [12]:
r = run_length(target_tokens, source_tokens)
s = edit_similarity(completion.text, split.continuation_text)

print("Run length:", r)
print("Edit similarity:", s)

Run length: 1
Edit similarity: 0.3092105263157895


In [13]:
r, p_rlb = rlb_score(
    target_tokens=target_tokens,
    source_tokens=source_tokens,
    reference_model=reference,
    prefix_token_ids=prefix_tokens,
)

s, p_esb = esb_score(
    target_text=completion.text,
    target_tokens=target_tokens,
    source_text=split.continuation_text,
    reference_model=reference,
    prefix_token_ids=prefix_tokens,
)

print("RLB:")
print("run_length:", r)
print("p_rlb:", p_rlb)

print("\nESB:")
print("edit_similarity:", s)
print("p_esb:", p_esb)

RLB:
run_length: 1
p_rlb: 0.06420686841011047

ESB:
edit_similarity: 0.3092105263157895
p_esb: 1.956252004837689e-33


In [ ]:
records = prepare_records(
    dataset_name="booktection",
    n=10,
    random_state=7,
    balance_labels=True,
)

results = []

for record in tqdm(records):
    split = split_by_tokens(
        record["text"],
        tokenizer=target.tokenizer,
        prefix_len=64,
        continuation_len=64,
    )

    completion = target.complete(
        split.prefix_text,
        max_new_tokens=64,
        do_sample=False,
    )

    target_tokens = completion.token_ids
    source_tokens = split.continuation_token_ids
    prefix_tokens = split.prefix_token_ids

    r, p_rlb = rlb_score(
        target_tokens=target_tokens,
        source_tokens=source_tokens,
        reference_model=reference,
        prefix_token_ids=prefix_tokens,
    )

    s, p_esb = esb_score(
        target_text=completion.text,
        target_tokens=target_tokens,
        source_text=split.continuation_text,
        reference_model=reference,
        prefix_token_ids=prefix_tokens,
    )

    results.append({
        "id": record["id"],
        "label": record["label"],
        "membership": record["estimated_membership"],
        "run_length": r,
        "p_rlb": p_rlb,
        "edit_similarity": s,
        "p_esb": p_esb,
        "prefix": split.prefix_text,
        "ground_truth": split.continuation_text,
        "target_completion": completion.text,
    })

df_results = pd.DataFrame(results)
df_results

100%|██████████| 10/10 [00:05<00:00,  1.99it/s]


,id,label,membership,run_length,p_rlb,edit_similarity,p_esb,prefix,ground_truth,target_completion
0,booktection_05069_A.txt,1,member,2,0.026349,0.289474,0.000000e+00,"Matilda didn’t trust herself to answer him, so...",had never seen. If only they would read a lit...,"had never had. They had always been happy, an..."
1,booktection_05840_A.txt,1,member,1,0.000380,0.271186,0.000000e+00,"“Not he,” said the Dwarf. “I saw the face and ...",scarce in camp. And there’s good eating on a ...,"scarce, and I don’t think you’ll find any in ..."
2,booktection_15994_A.txt,0,non_member,1,0.745633,0.284247,9.294050e-29,That must be why Im here. Base Paired it is. H...,"are my former schoolteachers, and even then o...",are the ones who know me personally. I have a...
3,booktection_11951_A.txt,0,non_member,0,1.000000,0.233463,0.000000e+00,Maybe hes getting lazy. Do you mind if we wait...,? The sheer stupidity of this investigation is...,on him? Peter looked at her in surprise. Ease...
4,booktection_15836_A.txt,0,non_member,0,1.000000,0.219512,1.257394e-32,"So much so, that when Camille’s shift came to ...",seemed as if they grew right out of the stone...,was hard to believe that the city was still a...
5,booktection_09976_A.txt,1,member,0,1.000000,0.260377,1.252216e-22,We had a good laugh at Waiting for Godot . Ni...,"that’s how things are. Fish, chips, pickled o...",I am a racing cyclist. I have been a cyclist ...
6,booktection_01944_A.txt,1,member,1,0.239796,0.248980,1.594529e-24,When Dunstan Cass turned his back on the cotta...,"ease, free from the presentiment of change. T...","ease. He had been a long way from home, and h..."
7,booktection_12324_A.txt,0,non_member,0,1.000000,0.197026,0.000000e+00,Simon couldnt help but look back at her and se...,question about her marriage date. And Simon w...,"door, but she did come out to the porch to sa..."
8,booktection_09760_A.txt,1,member,0,1.000000,0.245614,9.775113e-13,Christopher Abaddon. He is an invited guest of...,elevator on the right. It goes all the way up...,bathroom. He is a guest of Mr. Peter Solomon....
9,booktection_10461_A.txt,0,non_member,1,0.164745,0.287449,9.176547e-13,"Wulf said, Was I really the only one who lived...","of Fire, a plague that beset the people of Yi...",of the Sea. The Plague of fandauth is spreadi...


In [ ]:
out_dir = PROJECT_ROOT / "results"
out_dir.mkdir(exist_ok=True)

out_path = out_dir / "smoke_test_booktection_qwen05_target_qwen05_reference.csv"
df_results.to_csv(out_path, index=False)

print(out_path)

/home/alumno1/Desktop/NLP_Proyecto_Final-main/results/smoke_test_booktection_qwen05_target_qwen05_reference.csv


In [16]:
df_results[[
    "label",
    "run_length",
    "edit_similarity",
    "p_rlb",
    "p_esb"
]].groupby("label").mean()

,run_length,edit_similarity,p_rlb,p_esb
label,,,,
0,0.4,0.244339,0.782076,1.835309e-13
1,0.8,0.263126,0.453305,1.955023e-13
